# LeetCode 1437 — Check If All 1's Are at Least Length K Places Away bằng Python

**Link đề:** https://leetcode.com/problems/check-if-all-1s-are-at-least-length-k-places-away/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def kLengthApart(self, nums: list[int], k: int) -> bool:
        ...
```

### Mục tiêu học
1. Hiểu chính xác "cách nhau ít nhất `k` chỗ" = **số số 0 nằm giữa** hai số 1 liên tiếp `>= k`, tức `j - i - 1 >= k`.
2. Nắm kỹ thuật **nhớ vị trí lần xuất hiện trước** (`prev`) khi duyệt mảng một lượt — một pattern dùng rất nhiều.
3. Đi qua 4 lời giải: kiểm tra cửa sổ phía sau từng số 1 → danh sách vị trí → một lượt với `prev` → đếm số 0 liên tiếp; thêm 1 cách "mẹo chuỗi".
4. Trace từng bước, chứng minh bằng invariant, phân tích Time/Space.
5. Luyện biến thể: khoảng cách lớn nhất/nhỏ nhất giữa hai số 1, bài *Maximize Distance to Closest Person*.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng nhị phân `nums` (chỉ gồm 0 và 1) và số nguyên `k`. Trả về `true` nếu **mọi** số 1 đều cách nhau **ít nhất `k` chỗ**, ngược lại trả `false`.

| Ví dụ | `nums` | `k` | Output | Vì sao |
|---|---|---:|---|---|
| 1 | `[1,0,0,0,1,0,0,1]` | 2 | `true` | giữa các cặp 1 liên tiếp có 3 và 2 số 0, đều ≥ 2 |
| 2 | `[1,0,0,1,0,1]` | 2 | `false` | số 1 thứ hai và thứ ba chỉ cách nhau 1 số 0 |

**Constraints**
- `1 <= nums.length <= 10^5`
- `0 <= k <= nums.length`
- `nums[i]` là `0` hoặc `1`

Câu hỏi gợi mở:
- "Cách nhau `k` chỗ" đo bằng hiệu chỉ số `j - i` hay bằng số phần tử **nằm giữa** `j - i - 1`? (Ví dụ 2 trả lời giúp ta.)
- Có cần so **mọi cặp** số 1 không, hay chỉ cặp **liên tiếp**?
- Mảng không có số 1 nào, hoặc chỉ có một số 1, thì sao?

In [1]:
nums1, k1 = [1, 0, 0, 0, 1, 0, 0, 1], 2
nums2, k2 = [1, 0, 0, 1, 0, 1], 2

ones2 = [i for i, x in enumerate(nums2) if x == 1]
print("Vị trí các số 1 ở ví dụ 2:", ones2)
for a, c in zip(ones2, ones2[1:]):
    print(f"  cặp ({a},{c}): hiệu chỉ số = {c - a}, số 0 ở giữa = {c - a - 1}")

Vị trí các số 1 ở ví dụ 2: [0, 3, 5]
  cặp (0,3): hiệu chỉ số = 3, số 0 ở giữa = 2
  cặp (3,5): hiệu chỉ số = 2, số 0 ở giữa = 1


Cặp `(3, 5)` có hiệu chỉ số `2` nhưng chỉ có **1** số 0 ở giữa, và đề nói ví dụ 2 là `false` với `k = 2`. Vậy khoảng cách tính bằng **số phần tử nằm giữa**: điều kiện đúng là

$$j - i - 1 \ge k \quad\Longleftrightarrow\quad j - i > k.$$

Đây là chỗ dễ sai nhất của bài (lệch 1 — "off-by-one").

## 2. Nền tảng Python cần dùng

### 2.1. `enumerate` — duyệt cả chỉ số lẫn giá trị

`for i, x in enumerate(nums)` sinh lần lượt `(0, nums[0]), (1, nums[1]), ...`. Ta cần chỉ số `i` để tính khoảng cách.

In [2]:
for i, x in enumerate([1, 0, 1]):
    print("i =", i, "| x =", x)

i = 0 | x = 1
i = 1 | x = 0
i = 2 | x = 1


### 2.2. List comprehension có điều kiện

`[i for i, x in enumerate(nums) if x == 1]` đọc là: "lấy `i` với mỗi cặp `(i, x)`, **chỉ khi** `x == 1`". Kết quả là danh sách vị trí các số 1.

In [3]:
nums = [0, 1, 0, 0, 1, 1]
ones = [i for i, x in enumerate(nums) if x == 1]
print(ones)          # [1, 4, 5]
print(len(ones))     # 3 số 1

[1, 4, 5]
3


### 2.3. `zip(a, a[1:])` — duyệt các cặp liên tiếp

`a[1:]` là **slicing**: bỏ phần tử đầu. `zip` ghép phần tử thứ `t` của hai dãy, dừng khi dãy ngắn hơn hết. Vậy `zip(a, a[1:])` cho các cặp `(a[0],a[1]), (a[1],a[2]), ...`.

In [4]:
a = [1, 4, 5, 9]
print(a[1:])
print(list(zip(a, a[1:])))

[4, 5, 9]
[(1, 4), (4, 5), (5, 9)]


### 2.4. `all(...)` và `float('-inf')`

- `all(điều_kiện for ...)` trả `True` nếu **mọi** điều kiện đúng (và `True` với dãy rỗng!). Nó dừng ngay khi gặp `False` đầu tiên.
- `float('-inf')` là "âm vô cực": nhỏ hơn mọi số. Dùng làm giá trị khởi tạo cho "vị trí số 1 trước đó" khi **chưa có** số 1 nào — khi đó `i - prev - 1` luôn rất lớn, không bao giờ vi phạm.

In [5]:
print(all(x > 0 for x in [3, 5, 1]))    # True
print(all(x > 0 for x in [3, -5, 1]))   # False
print(all(x > 0 for x in []))           # True — dãy rỗng
prev = float('-inf')
print(7 - prev - 1, (7 - prev - 1) >= 100000)

True
False
True
inf True


### 2.5. Vì sao chỉ cần xét các cặp số 1 **liên tiếp**?

Giả sử vị trí các số 1 là `p0 < p1 < p2 < ...`. Với hai số 1 bất kỳ `p_a < p_c`, giữa chúng chứa cả đoạn `p_a..p_(a+1)`, nên số phần tử ở giữa `p_c - p_a - 1 >= p_(a+1) - p_a - 1`. Nếu mọi cặp **liên tiếp** đã đủ xa thì mọi cặp khác càng xa hơn. Ngược lại nếu một cặp liên tiếp vi phạm thì đáp án đã là `false`.

→ Chỉ cần kiểm tra `n_1 - 1` cặp liên tiếp (với `n_1` = số lượng số 1), không phải mọi cặp.

In [6]:
ones = [0, 4, 7]
print("cặp liên tiếp:", [(c - a - 1) for a, c in zip(ones, ones[1:])])
print("cặp (0,7)    :", 7 - 0 - 1, "≥ min cặp liên tiếp =", min(c - a - 1 for a, c in zip(ones, ones[1:])))

cặp liên tiếp: [3, 2]
cặp (0,7)    : 6 ≥ min cặp liên tiếp = 2


## 3. Cách 1 — Brute force: so **mọi cặp** số 1

Cách nghĩ thẳng từ đề "**mọi** số 1 cách nhau ít nhất `k`": lấy danh sách vị trí số 1, rồi kiểm tra **từng cặp** `(a, c)` với `a < c`. Chưa cần biết mẹo "chỉ cặp liên tiếp".

In [7]:
def k_apart_all_pairs(nums, k):
    ones = [i for i, x in enumerate(nums) if x == 1]
    for a in range(len(ones)):
        for c in range(a + 1, len(ones)):
            if ones[c] - ones[a] - 1 < k:
                return False
    return True

print(k_apart_all_pairs(nums1, k1))   # True
print(k_apart_all_pairs(nums2, k2))   # False

True
False


### 3.1. Chi phí của brute force

Nếu có `m` số 1, số cặp là `m(m-1)/2` → `O(m²)`, tệ nhất `O(n²)` (ví dụ `k = 0` và mảng toàn 1: mọi cặp đều hợp lệ nên không dừng sớm). Với `n = 10^5` là khoảng `5·10^9` phép — quá chậm.

In [8]:
def count_pair_checks(nums, k):
    ones = [i for i, x in enumerate(nums) if x == 1]
    checks = 0
    for a in range(len(ones)):
        for c in range(a + 1, len(ones)):
            checks += 1
            if ones[c] - ones[a] - 1 < k:
                return False, checks
    return True, checks

for n in [10, 100, 1000]:
    print(f"n={n:>4}, toàn số 1, k=0 →", count_pair_checks([1] * n, 0))

n=  10, toàn số 1, k=0 → (True, 45)
n= 100, toàn số 1, k=0 → (True, 4950)
n=1000, toàn số 1, k=0 → (True, 499500)


### 3.2. Cách 2 — Nhìn `k` ô phía sau mỗi số 1

Một cách trực giác khác: với mỗi số 1 ở vị trí `i`, các ô `i+1, ..., i+k` **phải toàn là 0** (nếu tồn tại). Thấy số 1 trong cửa sổ đó → `False`.

`min(i + k + 1, n)` giữ cho cửa sổ không vượt cuối mảng.

In [9]:
def k_apart_window(nums, k):
    n = len(nums)
    for i in range(n):
        if nums[i] == 1:
            for j in range(i + 1, min(i + k + 1, n)):   # các ô i+1 .. i+k
                if nums[j] == 1:
                    return False
    return True

print(k_apart_window(nums1, k1))   # True
print(k_apart_window(nums2, k2))   # False

True
False


Nhìn qua tưởng là `O(n · k)` (hai vòng lồng), nhưng thật ra **`O(n)`**: nếu mảng hợp lệ thì cửa sổ của hai số 1 liên tiếp **không chồng nhau** (số 1 kế tiếp nằm sau `i + k`), nên mỗi ô bị quét tối đa một lần; nếu không hợp lệ thì hàm dừng ngay lần đầu thấy vi phạm. Đếm thử để xác nhận — số lần kiểm tra luôn `< n`:

In [10]:
def count_window_checks(nums, k):
    n, checks = len(nums), 0
    for i in range(n):
        if nums[i] == 1:
            for j in range(i + 1, min(i + k + 1, n)):
                checks += 1
                if nums[j] == 1:
                    return False, checks
    return True, checks

for n, k in [(1000, 0), (1000, 10), (1000, 999)]:
    nums_many = [1 if i % (k + 1) == 0 else 0 for i in range(n)]
    print(f"n={n}, k={k}: {count_window_checks(nums_many, k)}")

n=1000, k=0: (True, 0)
n=1000, k=10: (True, 909)
n=1000, k=999: (True, 999)


Bài học: **hai vòng lồng nhau không tự động là `O(n²)`** — phải đếm tổng công việc thực sự. Tuy vậy cách này vẫn "nhìn về phía trước" và lập luận độ phức tạp hơi tinh tế; ta muốn một cách **hiển nhiên** `O(n)` và chỉ nhìn mỗi ô một lần.

## 4. Nút thắt và insight

**Nút thắt:** cách 1 so **mọi cặp** dù chỉ cặp liên tiếp quan trọng (mục 2.5); cách 2 phải nhìn **tới trước** và cần lập luận phức tạp để thấy nó nhanh.

**Insight:** đảo hướng nhìn — khi gặp một số 1 ở vị trí `i`, ta chỉ cần biết **số 1 gần nhất phía trước** nằm ở đâu (`prev`). Mục 2.5 đã chứng minh chỉ cặp liên tiếp là quan trọng, và cặp liên tiếp của `i` chính là `(prev, i)`.

> Quy tắc: **Nếu** điều kiện chỉ liên quan tới cặp phần tử đặc biệt liên tiếp, **thì** duyệt một lượt và nhớ vị trí của phần tử đặc biệt gần nhất.

Thử tính tay trên ví dụ 2: vị trí số 1 là 0, 3, 5.

In [11]:
prev = None
for i, x in enumerate([1, 0, 0, 1, 0, 1]):
    if x == 1:
        if prev is not None:
            print(f"gặp 1 tại {i}, số 1 trước ở {prev}: số 0 ở giữa = {i - prev - 1}")
        prev = i

gặp 1 tại 3, số 1 trước ở 0: số 0 ở giữa = 2
gặp 1 tại 5, số 1 trước ở 3: số 0 ở giữa = 1


## 5. Cách 3 — Danh sách vị trí các số 1 (chỉ cặp liên tiếp)

Bước trung gian dễ hiểu: gom vị trí các số 1 vào list `ones`, rồi kiểm tra mọi cặp liên tiếp. Đúng và `O(n)`, nhưng tốn thêm `O(số lượng số 1)` bộ nhớ.

In [12]:
def k_apart_positions(nums, k):
    ones = [i for i, x in enumerate(nums) if x == 1]
    return all(c - a - 1 >= k for a, c in zip(ones, ones[1:]))

print(k_apart_positions(nums1, k1))   # True
print(k_apart_positions(nums2, k2))   # False
print(k_apart_positions([0, 0, 0], 3))  # True — không có số 1

True
False
True


## 6. Cách 4 (lời giải chính) — Một lượt với `prev`

### 6.1. Mổ xẻ từng đoạn

**Đoạn 1 — khởi tạo:** `prev = -k - 1`. Vì sao? Ta muốn số 1 **đầu tiên** luôn hợp lệ. Với `i >= 0`: `i - prev - 1 = i + k >= k`. ✓ (Dùng `float('-inf')` cũng được, nhưng một số nguyên gọn hơn và tránh trộn float.)

**Đoạn 2 — duyệt và kiểm tra:**
```python
for i, x in enumerate(nums):
    if x == 1:
        if i - prev - 1 < k:   # quá gần số 1 trước
            return False
        prev = i               # cập nhật SAU khi kiểm tra
```

**Đoạn 3 — không vi phạm lần nào:** `return True`.

Thứ tự "**kiểm tra trước, cập nhật sau**" giống Two Sum: nếu cập nhật `prev = i` trước, ta sẽ so `i` với chính nó.

In [13]:
def k_apart_prev(nums, k):
    prev = -k - 1                     # "số 1 ảo" đủ xa về bên trái
    for i, x in enumerate(nums):
        if x == 1:
            if i - prev - 1 < k:
                return False
            prev = i
    return True

print(k_apart_prev(nums1, k1))   # True
print(k_apart_prev(nums2, k2))   # False

True
False


## 7. TRACE từng bước

### 7.1. Ví dụ 2: `nums = [1,0,0,1,0,1]`, `k = 2`, `prev` khởi tạo `-3`

| i | x | `prev` trước | số 0 ở giữa `i-prev-1` | `< k`? | Hành động |
|---:|---:|---:|---:|---|---|
| 0 | 1 | -3 | 2 | không | `prev = 0` |
| 1 | 0 | 0 | – | – | bỏ qua |
| 2 | 0 | 0 | – | – | bỏ qua |
| 3 | 1 | 0 | 2 | không | `prev = 3` |
| 4 | 0 | 3 | – | – | bỏ qua |
| 5 | 1 | 3 | **1** | **có** | **return False** |

In [14]:
def trace_k_apart(nums, k):
    prev = -k - 1
    print(f"nums={nums}, k={k}, prev khởi tạo={prev}")
    for i, x in enumerate(nums):
        if x == 0:
            print(f"  i={i} x=0 → bỏ qua")
            continue
        gap = i - prev - 1
        if gap < k:
            print(f"  i={i} x=1 prev={prev} gap={gap} < k → FALSE")
            return False
        print(f"  i={i} x=1 prev={prev} gap={gap} >= k → prev = {i}")
        prev = i
    print("  hết mảng, không vi phạm → TRUE")
    return True

trace_k_apart([1, 0, 0, 1, 0, 1], 2)

nums=[1, 0, 0, 1, 0, 1], k=2, prev khởi tạo=-3
  i=0 x=1 prev=-3 gap=2 >= k → prev = 0
  i=1 x=0 → bỏ qua
  i=2 x=0 → bỏ qua
  i=3 x=1 prev=0 gap=2 >= k → prev = 3
  i=4 x=0 → bỏ qua
  i=5 x=1 prev=3 gap=1 < k → FALSE


False

`continue` nhảy ngay sang vòng lặp kế tiếp, bỏ qua phần code còn lại trong thân vòng — giúp in gọn trường hợp `x = 0`.

### 7.2. Ví dụ 1 và các nhánh đặc biệt `k = 0`, hai số 1 kề nhau

In [15]:
trace_k_apart([1, 0, 0, 0, 1, 0, 0, 1], 2)
print()
trace_k_apart([1, 1], 0)     # k=0: kề nhau vẫn hợp lệ
print()
trace_k_apart([1, 1], 1)     # k=1: kề nhau vi phạm

nums=[1, 0, 0, 0, 1, 0, 0, 1], k=2, prev khởi tạo=-3
  i=0 x=1 prev=-3 gap=2 >= k → prev = 0
  i=1 x=0 → bỏ qua
  i=2 x=0 → bỏ qua
  i=3 x=0 → bỏ qua
  i=4 x=1 prev=0 gap=3 >= k → prev = 4
  i=5 x=0 → bỏ qua
  i=6 x=0 → bỏ qua
  i=7 x=1 prev=4 gap=2 >= k → prev = 7
  hết mảng, không vi phạm → TRUE

nums=[1, 1], k=0, prev khởi tạo=-1
  i=0 x=1 prev=-1 gap=0 >= k → prev = 0
  i=1 x=1 prev=0 gap=0 >= k → prev = 1
  hết mảng, không vi phạm → TRUE

nums=[1, 1], k=1, prev khởi tạo=-2
  i=0 x=1 prev=-2 gap=1 >= k → prev = 0
  i=1 x=1 prev=0 gap=0 < k → FALSE


False

Với `k = 0`, gap luôn `>= 0` nên đáp án luôn `True` — điều này khớp trực giác "cách nhau ít nhất 0 chỗ" luôn đúng.

## 8. Cách 5 — Đếm số 0 kể từ số 1 trước

Thay vì nhớ vị trí, nhớ **số số 0 đã gặp kể từ số 1 trước** (`zeros`). Gặp 1: nếu đã có số 1 trước và `zeros < k` → `False`; sau đó đặt lại `zeros = 0`. Gặp 0: `zeros += 1`.

Khởi tạo `zeros = k` để số 1 đầu tiên luôn qua (tương tự "số 1 ảo").

In [16]:
def k_apart_zeros(nums, k):
    zeros = k                 # coi như trước số 1 đầu tiên đã có đủ k số 0
    for x in nums:
        if x == 1:
            if zeros < k:
                return False
            zeros = 0
        else:
            zeros += 1
    return True

print(k_apart_zeros(nums1, k1), k_apart_zeros(nums2, k2))

True False


## 9. Cách 6 (mẹo sáng tạo) — Cắt chuỗi theo ký tự `'1'`

Ghép mảng thành chuỗi `"10001001"`, rồi `split('1')` → các đoạn số 0 nằm **giữa** các số 1 là các phần tử ở giữa danh sách (bỏ phần tử đầu và cuối, vì đó là số 0 trước số 1 đầu / sau số 1 cuối).

In [17]:
s = "".join(map(str, [1, 0, 0, 1, 0, 1]))
print(repr(s))
parts = s.split("1")
print(parts)               # ['', '00', '0', '']
print("đoạn giữa:", parts[1:-1], "→ độ dài", [len(p) for p in parts[1:-1]])

'100101'
['', '00', '0', '']
đoạn giữa: ['00', '0'] → độ dài [2, 1]


In [18]:
def k_apart_split(nums, k):
    parts = "".join(map(str, nums)).split("1")
    return all(len(p) >= k for p in parts[1:-1])

print(k_apart_split(nums1, k1), k_apart_split(nums2, k2), k_apart_split([0, 0], 5), k_apart_split([1], 1))

True False True True


- `map(str, nums)` áp `str` lên từng phần tử → `'1','0',...`; `"".join(...)` nối thành một chuỗi.
- Không có số 1: `split` trả `['00']`, `parts[1:-1] = []` → `all([]) = True`. ✓
- Cách này hay để luyện tư duy, nhưng tốn `O(n)` bộ nhớ cho chuỗi, nên bản nộp chính vẫn là cách 4.

## 10. Code nộp LeetCode (Python 3)

In [19]:
class Solution:
    def kLengthApart(self, nums: list[int], k: int) -> bool:
        prev = -k - 1
        for i, x in enumerate(nums):
            if x == 1:
                if i - prev - 1 < k:
                    return False
                prev = i
        return True

sol = Solution()
print(sol.kLengthApart([1, 0, 0, 0, 1, 0, 0, 1], 2))   # True
print(sol.kLengthApart([1, 0, 0, 1, 0, 1], 2))         # False

True
False


## 11. Vì sao thuật toán đúng?

**Invariant:** ở đầu mỗi vòng `i`, `prev` là vị trí số 1 **gần nhất bên trái** `i` (hoặc `-k-1` nếu chưa có), và mọi cặp số 1 liên tiếp nằm hoàn toàn trong `nums[0..i-1]` đều cách nhau `>= k`.

- **Khởi tạo:** chưa có cặp nào → đúng hiển nhiên; `prev = -k-1` là "số 1 ảo" không ảnh hưởng (gap của số 1 thật đầu tiên `= i + k >= k`).
- **Duy trì:** nếu `nums[i] = 1`, cặp liên tiếp mới duy nhất là `(prev, i)`. Ta kiểm tra đúng cặp đó; qua được thì cập nhật `prev = i` → invariant còn đúng cho `i+1`. Nếu `nums[i] = 0`, không có cặp mới, `prev` giữ nguyên.
- **Kết thúc:** nếu hết vòng mà không trả `False`, mọi cặp liên tiếp đều đạt; theo mục 2.5 mọi cặp bất kỳ cũng đạt → `True` đúng. Nếu trả `False`, ta đã chỉ ra một cặp cụ thể vi phạm → đúng.

## 12. Phân tích Time / Space

Gọi `n = len(nums)`, `m` = số lượng số 1.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi cặp | `O(n + m²)`, tệ nhất `O(n²)` | `O(m)` | brute force |
| 2. Cửa sổ phía sau | `O(n)` (cửa sổ không chồng nhau) | `O(1)` | nhìn về trước |
| 3. List vị trí | `O(n)` | `O(m)` | dễ hiểu |
| 4. `prev` một lượt | `O(n)` | `O(1)` | **khuyên dùng** |
| 5. Đếm số 0 | `O(n)` | `O(1)` | tương đương cách 4 |
| 6. Split chuỗi | `O(n)` | `O(n)` | mẹo, tốn bộ nhớ |

Cả cách 4 và 5 có thể dừng sớm ngay khi gặp vi phạm đầu tiên.

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.kLengthApart([1,0,0,0,1,0,0,1], 2), True)
check("ví dụ 2", sol.kLengthApart([1,0,0,1,0,1], 2), False)
check("không có số 1", sol.kLengthApart([0,0,0], 3), True)
check("một số 1, k = n", sol.kLengthApart([0,1,0], 3), True)
check("k = 0, toàn 1", sol.kLengthApart([1,1,1,1], 0), True)
check("kề nhau, k = 1", sol.kLengthApart([1,1], 1), False)
check("đúng bằng k (biên)", sol.kLengthApart([1,0,0,1], 2), True)
check("thiếu 1 (off-by-one)", sol.kLengthApart([1,0,0,1], 3), False)
check("vi phạm ở cuối", sol.kLengthApart([1,0,0,0,1,0,1], 2), False)

PASS  ví dụ 1: True
PASS  ví dụ 2: False
PASS  không có số 1: True
PASS  một số 1, k = n: True
PASS  k = 0, toàn 1: True
PASS  kề nhau, k = 1: False
PASS  đúng bằng k (biên): True
PASS  thiếu 1 (off-by-one): False
PASS  vi phạm ở cuối: False


### 13.1. Đối chiếu 6 cách — vét cạn mọi mảng độ dài ≤ 10 và mọi `k`

Có `2^1 + ... + 2^10 ≈ 2046` mảng nhị phân. Oracle (đáp án tin cậy) là cách 1 — kiểm tra **mọi cặp**, không dùng mẹo "cặp liên tiếp".

In [21]:
from itertools import product

oracle = k_apart_all_pairs
methods = [k_apart_window, k_apart_positions, k_apart_prev, k_apart_zeros, k_apart_split]
total = 0
for n in range(1, 11):
    for bits in product([0, 1], repeat=n):
        nums = list(bits)
        for k in range(0, n + 1):
            exp = oracle(nums, k)
            for f in methods:
                assert f(nums, k) == exp, (f.__name__, nums, k)
            total += 1
print("Đã kiểm tra", total, "bộ (nums, k): 5 cách còn lại đều khớp oracle")

Đã kiểm tra 20480 bộ (nums, k): 5 cách còn lại đều khớp oracle


`itertools.product([0, 1], repeat=n)` sinh mọi bộ `n` phần tử lấy từ `{0, 1}` — tức mọi mảng nhị phân độ dài `n`.

### 13.2. Test ngẫu nhiên có seed với mảng dài

In [22]:
import random
rng = random.Random(1437)
for _ in range(200):
    n = rng.randint(1, 2000)
    p = rng.choice([0.01, 0.05, 0.3])
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    k = rng.randint(0, 60)
    exp = k_apart_positions(nums, k)
    assert k_apart_prev(nums, k) == exp == k_apart_zeros(nums, k) == k_apart_split(nums, k)
print("200 mảng ngẫu nhiên: PASS")

200 mảng ngẫu nhiên: PASS


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Dùng hiệu chỉ số thay cho số phần tử ở giữa (off-by-one)

```python
if i - prev < k:      # sai: phải là i - prev - 1 < k
```
**Vì sao sai:** `[1,0,0,1]`, `k=3`: hiệu chỉ số = 3 → kết luận "đủ xa", nhưng thật ra chỉ có 2 số 0 ở giữa.

In [23]:
def wrong_off_by_one(nums, k):
    prev = -k - 1
    for i, x in enumerate(nums):
        if x == 1:
            if i - prev < k:
                return False
            prev = i
    return True

print("Sai:", wrong_off_by_one([1, 0, 0, 1], 3), "| Đúng:", k_apart_prev([1, 0, 0, 1], 3))

Sai: True | Đúng: False


### Lỗi 2 — Khởi tạo `prev = 0` (hoặc `-1`)

`prev = 0` ngầm giả sử có số 1 ở vị trí 0. Với `[0,1]`, `k=1`: gap `= 1-0-1 = 0 < 1` → trả `False` sai (chỉ có một số 1!). **Sửa:** `prev = -k-1`, hoặc `None` và bỏ qua lần kiểm tra đầu.

In [24]:
def wrong_prev_zero(nums, k):
    prev = 0
    for i, x in enumerate(nums):
        if x == 1:
            if i - prev - 1 < k:
                return False
            prev = i
    return True

print("Sai:", wrong_prev_zero([0, 1], 1), "| Đúng:", k_apart_prev([0, 1], 1))

Sai: False | Đúng: True


Tệ hơn: nếu `nums[0] = 1` thì ở `i = 0` gap `= 0 - 0 - 1 = -1 < k` với **mọi** `k >= 0`, kể cả `k = 0`. Vậy `[1]`, `k = 0` bị trả `False`, trong khi đáp án hiển nhiên là `True`:

In [25]:
print("Sai:", wrong_prev_zero([1], 0), "| Đúng:", k_apart_prev([1], 0))

Sai: False | Đúng: True


### Lỗi 3 — Cập nhật `prev` trước khi kiểm tra

```python
if x == 1:
    prev = i
    if i - prev - 1 < k:   # luôn là -1 < k
```
**Vì sao sai:** so sánh số 1 với chính nó → gap `= -1`, sai với mọi `k >= 0`.

In [26]:
def wrong_update_first(nums, k):
    prev = -k - 1
    for i, x in enumerate(nums):
        if x == 1:
            prev = i
            if i - prev - 1 < k:
                return False
    return True

print("Sai:", wrong_update_first([1, 0, 0, 0, 1], 2), "| Đúng:", k_apart_prev([1, 0, 0, 0, 1], 2))

Sai: False | Đúng: True


### Lỗi 4 — Cắt chuỗi nhưng quên bỏ hai đoạn đầu/cuối

`"0010".split("1") = ['00', '0']`: đoạn đầu là các số 0 **trước** số 1 đầu tiên, không nằm giữa hai số 1. Nếu kiểm tra cả đoạn này với `k=3` sẽ trả `False` sai.

In [27]:
def wrong_split(nums, k):
    return all(len(p) >= k for p in "".join(map(str, nums)).split("1"))

print("Sai:", wrong_split([0, 0, 1, 0], 3), "| Đúng:", k_apart_split([0, 0, 1, 0], 3))

Sai: False | Đúng: True


## 15. Mẹo nhận dạng pattern

- Đề hỏi về **khoảng cách giữa các phần tử đặc biệt liên tiếp** (số 1, ký tự trùng, ghế có người…) → duyệt một lượt, nhớ **`prev`** = vị trí phần tử đặc biệt gần nhất.
- "Cách nhau ít nhất `k`" thường cần **tự định nghĩa lại** bằng chỉ số và kiểm tra trên ví dụ đề để tránh lệch 1.
- Giá trị khởi tạo "ảo" (`-k-1`, `-inf`) giúp xóa trường hợp đặc biệt "chưa có phần tử trước".
- Bài cùng họ: *Maximize Distance to Closest Person* (849), *Shortest Distance to a Character* (821), *Binary Gap* (868), *Contains Duplicate II* (219 — nhớ vị trí gần nhất của từng giá trị).

## 16. Biến thể

### 16.1. Khoảng cách nhỏ nhất giữa hai số 1 liên tiếp (số 0 ở giữa)
Trả về `min` gap; nếu ít hơn 2 số 1 trả `-1`. Bài gốc tương đương với `min_gap >= k`.

In [28]:
def min_gap(nums):
    prev, best = None, None
    for i, x in enumerate(nums):
        if x == 1:
            if prev is not None:
                gap = i - prev - 1
                best = gap if best is None else min(best, gap)
            prev = i
    return -1 if best is None else best

print(min_gap([1, 0, 0, 0, 1, 0, 0, 1]))   # 2
print(min_gap([1, 0, 0, 1, 0, 1]))         # 1
print(min_gap([0, 1, 0]))                  # -1

2
1
-1


### 16.2. Binary Gap (LeetCode 868) — khoảng cách **chỉ số** lớn nhất giữa hai bit 1 liên tiếp của số `n`
Cùng kỹ thuật `prev`, chỉ khác dữ liệu là các bit của một số và đo bằng hiệu chỉ số.

In [29]:
def binary_gap(n):
    bits = bin(n)[2:]            # bin(22) = '0b10110' → bỏ '0b'
    prev, best = None, 0
    for i, ch in enumerate(bits):
        if ch == "1":
            if prev is not None:
                best = max(best, i - prev)
            prev = i
    return best

print(bin(22), binary_gap(22))   # 2
print(bin(8), binary_gap(8))     # 0

0b10110 2
0b1000 0


## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** `nums = [0,1,0,0,0,0,1,0,1]`. Liệt kê vị trí số 1, gap giữa các cặp liên tiếp. Đáp án với `k = 1`, `k = 2`?

**Bài 2 — Dự đoán output.**
```python
print("100101".split("1"))
print(all([]), any([]))
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải `prev`

In [30]:
def exercise_k_apart(nums, k):
    # TODO 1: khởi tạo prev "ảo" đủ xa
    # TODO 2: gặp số 1 -> kiểm tra gap = i - prev - 1, rồi mới cập nhật prev
    pass

# assert exercise_k_apart([1,0,0,0,1,0,0,1], 2) == True
# assert exercise_k_apart([1,0,0,1,0,1], 2) == False

### Bài 4 — TODO: `k` lớn nhất có thể
Trả về `k` lớn nhất sao cho `kLengthApart(nums, k)` vẫn `True`. Nếu có ít hơn 2 số 1, trả `len(nums)` (mọi `k` hợp lệ theo constraints đều đúng).

In [31]:
def exercise_max_k(nums):
    # TODO: liên hệ với min_gap
    pass

# assert exercise_max_k([1,0,0,0,1,0,0,1]) == 2
# assert exercise_max_k([0,1,0]) == 3

### Bài 5 — TODO: Maximize Distance to Closest Person (LeetCode 849)
`seats` gồm 0 (trống) và 1 (có người), có ít nhất một 0 và một 1. Chọn ghế trống sao cho khoảng cách (hiệu chỉ số) tới người gần nhất là lớn nhất. Trả về khoảng cách đó.

In [32]:
def exercise_max_dist_to_closest(seats):
    # TODO: 3 trường hợp — đoạn 0 ở đầu, đoạn 0 ở giữa hai số 1, đoạn 0 ở cuối
    pass

# assert exercise_max_dist_to_closest([1,0,0,0,1,0,1]) == 2
# assert exercise_max_dist_to_closest([1,0,0,0]) == 3
# assert exercise_max_dist_to_closest([0,1]) == 1

### Bài 6 — Mini challenge: số số 1 tối đa có thể thêm vào
Cho `nums` đã thỏa điều kiện `k`. Có thể đổi bao nhiêu số 0 thành 1 (nhiều nhất) mà điều kiện vẫn thỏa?

In [33]:
def exercise_max_add_ones(nums, k):
    # TODO: tham lam từ trái sang phải — đặt 1 vào ô 0 sớm nhất có thể,
    #       nhưng phải kiểm tra cả số 1 gần nhất BÊN PHẢI
    pass

# assert exercise_max_add_ones([1,0,0,0,0,0,1], 1) == 2      # 1 0 1 0 1 0 1
# assert exercise_max_add_ones([0,0,0], 1) == 2
# assert exercise_max_add_ones([1,0,1], 1) == 0

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Chỉ cặp liên tiếp quan trọng. | 1 vòng `enumerate`, biến `prev`. | `prev=-k-1`; `if x==1: if i-prev-1<k: return False; prev=i` |
| 4 | `True` ⇔ `k <= min gap`. | dùng lại `min_gap`. | `g = min_gap(nums); return len(nums) if g == -1 else g` |
| 5 | Đoạn 0 giữa dài `L` cho khoảng `(L+1)//2`; đoạn ở mép cho `L`. | duyệt với `prev`, xử lý riêng đầu/cuối. | đầu: `first_one`; giữa: `(i-prev)//2`; cuối: `n-1-last_one` |
| 6 | Đặt 1 càng sớm càng tốt là tối ưu. | cần vị trí số 1 gần nhất bên trái (`prev`) và bên phải (`next_one[i]`). | đặt tại `i` nếu `i-prev-1>=k` và `next_one[i]-i-1>=k`, rồi `prev=i` |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** số 1 ở `1, 6, 8`; gap `4` và `1` → `k=1`: `True`, `k=2`: `False`.
**Bài 2:** `['', '00', '0', '']`, rồi `True False`.

In [34]:
def answer_k_apart(nums, k):
    prev = -k - 1
    for i, x in enumerate(nums):
        if x == 1:
            if i - prev - 1 < k:
                return False
            prev = i
    return True

def answer_max_k(nums):
    g = min_gap(nums)
    return len(nums) if g == -1 else g

def answer_max_dist_to_closest(seats):
    n = len(seats)
    prev, best = None, 0
    for i, x in enumerate(seats):
        if x == 1:
            if prev is None:
                best = max(best, i)                 # đoạn 0 ở đầu
            else:
                best = max(best, (i - prev) // 2)   # đoạn 0 ở giữa
            prev = i
    best = max(best, n - 1 - prev)                  # đoạn 0 ở cuối
    return best

def answer_max_add_ones(nums, k):
    n = len(nums)
    INF = 10**9
    next_one = [INF] * (n + 1)
    for i in range(n - 1, -1, -1):
        next_one[i] = i if nums[i] == 1 else next_one[i + 1]
    prev, added = -k - 1, 0
    for i in range(n):
        if nums[i] == 1:
            prev = i
        elif i - prev - 1 >= k and next_one[i] - i - 1 >= k:
            added += 1
            prev = i
    return added

assert answer_k_apart([1,0,0,0,1,0,0,1], 2) == True
assert answer_k_apart([1,0,0,1,0,1], 2) == False
assert answer_max_k([1,0,0,0,1,0,0,1]) == 2
assert answer_max_k([0,1,0]) == 3
assert answer_max_dist_to_closest([1,0,0,0,1,0,1]) == 2
assert answer_max_dist_to_closest([1,0,0,0]) == 3
assert answer_max_dist_to_closest([0,1]) == 1
assert answer_max_add_ones([1,0,0,0,0,0,1], 1) == 2
assert answer_max_add_ones([0,0,0], 1) == 2
assert answer_max_add_ones([1,0,1], 1) == 0
print("Đáp án bài 3–6: PASS")

Đáp án bài 3–6: PASS


In [35]:
# Kiểm chứng bài 6 bằng vét cạn trên mảng nhỏ
from itertools import combinations
def brute_add(nums, k):
    zeros = [i for i, x in enumerate(nums) if x == 0]
    for r in range(len(zeros), -1, -1):
        for comb in combinations(zeros, r):
            t = nums[:]
            for i in comb:
                t[i] = 1
            if answer_k_apart(t, k):
                return r
for n in range(1, 9):
    for bits in product([0, 1], repeat=n):
        for k in range(0, 4):
            if answer_k_apart(list(bits), k):
                assert answer_max_add_ones(list(bits), k) == brute_add(list(bits), k), (bits, k)
print("Bài 6 khớp vét cạn; bài 1:", answer_k_apart([0,1,0,0,0,0,1,0,1], 1), answer_k_apart([0,1,0,0,0,0,1,0,1], 2))
print("100101".split("1")); print(all([]), any([]))

Bài 6 khớp vét cạn; bài 1: True False
['', '00', '0', '']
True False


## 20. Template nhớ nhanh + checklist submit

```python
prev = -k - 1                       # phần tử đặc biệt "ảo" đủ xa
for i, x in enumerate(nums):
    if là_phần_tử_đặc_biệt(x):
        if khoảng_cách(prev, i) vi phạm:   # KIỂM TRA trước
            return False
        prev = i                    # CẬP NHẬT sau
return True
```

**Checklist trước khi nộp**
- [ ] Khoảng cách = số phần tử **ở giữa** = `i - prev - 1` (đã thử lại trên ví dụ 2).
- [ ] `prev` khởi tạo không tạo ra "số 1 giả" gần số 1 đầu tiên.
- [ ] Kiểm tra trước, cập nhật `prev` sau.
- [ ] `k = 0`, mảng không có số 1, chỉ một số 1 → `True`.
- [ ] Method tên `kLengthApart`, trả về `bool`.